# Kronos Multi-Timeframe Benchmark (M1, M5, M15, H1, H4) — Google Colab GPU

**فقط برای پژوهش در Google Colab (بدون اتصال به MT5، بدون ثبت سفارش، بدون ساخت سرور عمومی).**

این نوت‌بوک مدل رسمی **`shiyu-coder/Kronos`** (پین‌شده روی کامیت `67b630e67f6a18c9e9be918d9b4337c960db1e9a`) را روی فایل دیتای تمیز و بدون اطلاعات حسابِ خودتان (`kronos_multitf_XAUUSD_i.json`) برای هر ۵ تایم‌فریم **`M1`, `M5`, `M15`, `H1`, `H4`** اجرا می‌کند:
- **۴۰ نقطه تاریخی کاملاً پشت‌سرهم و بدون هم‌پوشانی** در هر تایم‌فریم (۲۰۰ پیش‌بینی رسمی Kronos، نیمه اول در برابر نیمه دوم).
- **مقایسه خطا (`MAE`) با خط پایه `No-Change`** روی افق‌های `+1`, `+2`, `+5`, `+12` کندل.
- **ارزیابی واقعی معامله** (دقت جهت `1B / 2B / 5B / 12B`، سود/ضرر خالص بعد از اسپرد زنده آلپاری، `PF`، نیمه اول/دوم، تست فشار `1.5× اسپرد + $5`، و فیلتر سیگنال قوی `>= 2× spread`).

> **نکته مهم:** تا زمانی که واقعاً اجرا نشده، هیچ نتیجه‌ای ساخته یا فرض نمی‌شود. در Colab حتماً از منوی **Runtime → Change runtime type → T4 GPU** را انتخاب کنید (~۲ دقیقه اجرا).


In [ ]:
# CELL 1 — Colab-only setup (pinned official shiyu-coder/Kronos)
import os, sys, subprocess, json, math, time
from importlib import metadata as _metadata

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
assert IN_COLAB, 'This notebook must run only inside Google Colab runtime.'
print('Runtime Python:', sys.version.split()[0], '| executable:', sys.executable)
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError(
        'This pinned notebook supports Python 3.10-3.12 only. '
        'Select a Python 3.11/3.12 runtime in Colab.'
    )

KRONOS_COMMIT = '67b630e67f6a18c9e9be918d9b4337c960db1e9a'
WORK = '/content/kronos-multitf'
REPO = f'{WORK}/Kronos'
os.makedirs(WORK, exist_ok=True)

if not os.path.exists(REPO):
    subprocess.check_call(['git', 'clone', 'https://github.com/shiyu-coder/Kronos.git', REPO])
subprocess.check_call(['git', '-C', REPO, 'fetch', '--all'])
subprocess.check_call(['git', '-C', REPO, 'checkout', KRONOS_COMMIT])
head = subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD'], text=True).strip()
dirty = subprocess.check_output(['git', '-C', REPO, 'status', '--porcelain'], text=True).strip()
assert head == KRONOS_COMMIT and not dirty, 'Pinned clean Kronos checkout required'

# Install Kronos requirements (keep pip output visible for diagnostics)
install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', os.path.join(REPO, 'requirements.txt')])
if install.returncode:
    raise RuntimeError('Dependency installation failed. Read the pip ERROR lines above.')

if REPO not in sys.path:
    sys.path.insert(0, REPO)

import torch
import pandas as pd
assert torch.tensor([1.0, 2.0]).sum().item() == 3.0
from model import Kronos, KronosPredictor, KronosTokenizer

_KRONOS_RUNTIME_READY = True
print('Setup OK | commit:', head[:12], '| torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


In [ ]:
# CELL 2 — Upload redacted multi-TF OHLCV bundle (`logs/kronos_multitf_XAUUSD_i.json`)
assert globals().get('_KRONOS_RUNTIME_READY') is True, 'Run Cell 1 first.'
from google.colab import files

uploaded = files.upload()
assert uploaded, 'No file uploaded'
fname = next(iter(uploaded))
bundle = json.loads(uploaded[fname].decode('utf-8'))

forbidden = ('login', 'password', 'token', 'secret', 'account', 'journal', 'credential', 'balance', 'equity', 'server')
blob = json.dumps(bundle).lower()
found_bad = [k for k in forbidden if f'"{k}"' in blob]
assert not found_bad, f'Input appears to contain sensitive fields ({found_bad}); redact first.'
assert bundle.get('schema') == 'kronos-multitf-redacted-ohlcv/v1', 'Upload the bundle exported by export_kronos_multitf_bundle.py'

symbol = str(bundle['symbol'])
spread_px = float(bundle['spread_px'])
mult = float(bundle['mult'])
tf_map = bundle['timeframes']
print(f'Loaded redacted bundle: symbol={symbol} spread={spread_px} mult={mult}')
for k, rows in tf_map.items():
    print(f'  {k:3s}: {len(rows):5d} closed bars')


In [ ]:
# CELL 3 — Run official KronosPredictor across M1, M5, M15, H1, H4 & print full table
assert globals().get('_KRONOS_RUNTIME_READY') is True, 'Run Cell 1 first.'
assert 'bundle' in globals(), 'Run Cell 2 and upload your redacted JSON first.'

TOKENIZER_ID = os.environ.get('KRONOS_TOKENIZER_ID', 'NeoQuasar/Kronos-Tokenizer-base')
MODEL_ID = os.environ.get('KRONOS_MODEL_ID', 'NeoQuasar/Kronos-small')
CONTEXT_LEN = int(os.environ.get('KRONOS_CONTEXT_LEN', '256'))
PRED_LEN = int(os.environ.get('KRONOS_PRED_LEN', '12'))
N_ORIGINS = int(os.environ.get('KRONOS_ORIGINS_PER_TF', '40'))
SAMPLE_COUNT = int(os.environ.get('KRONOS_SAMPLE_COUNT', '3'))
LOT = 0.01

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Loading {MODEL_ID} + {TOKENIZER_ID} on {device} (origins/TF={N_ORIGINS}, samples={SAMPLE_COUNT})...')
tokenizer = KronosTokenizer.from_pretrained(TOKENIZER_ID)
model = Kronos.from_pretrained(MODEL_ID)
predictor = KronosPredictor(model, tokenizer, device=device, max_context=512)

TF_FREQ = {'M1': '1min', 'M5': '5min', 'M15': '15min', 'H1': '1h', 'H4': '4h'}
TF_SEC = {'M1': 60, 'M5': 300, 'M15': 900, 'H1': 3600, 'H4': 14400}


def _select_origins(bars, context_len=256, pred_len=12, n_origins=40):
    n = len(bars)
    first = context_len
    last = n - pred_len
    if last < first or n_origins <= 0:
        return []
    usable = last - first
    if usable == 0:
        return [first]
    max_non_overlap = (usable // pred_len) + 1
    count = min(n_origins, max_non_overlap)
    if count <= 1:
        return [first]
    step = max(pred_len, usable // (count - 1))
    out, pos = [], first
    while pos <= last and len(out) < count:
        out.append(pos)
        pos += step
    return out


def _summarize(trades):
    n = len(trades)
    if n == 0:
        return {'trades': 0, 'wins': 0, 'losses': 0, 'wr': None, 'net': 0.0, 'pf': None, 'max_dd': 0.0, 'excl_best': 0.0}
    pnls = [float(t['pnl']) for t in trades]
    wins = sum(1 for p in pnls if p > 0)
    losses = sum(1 for p in pnls if p < 0)
    gw = sum(p for p in pnls if p > 0)
    gl = abs(sum(p for p in pnls if p < 0))
    net = sum(pnls)
    pf = (gw / gl) if gl > 0 else (99.0 if gw > 0 else None)
    eq = peak = dd = 0.0
    for p in pnls:
        eq += p
        peak = max(peak, eq)
        dd = max(dd, peak - eq)
    return {
        'trades': n, 'wins': wins, 'losses': losses, 'wr': round(100.0 * wins / n, 2),
        'net': round(net, 2), 'pf': round(pf, 3) if pf is not None else None,
        'max_dd': round(dd, 2), 'excl_best': round(net - max(pnls), 2),
    }


def _eval_tf(tf_label, bars, origin_results, spread_px, mult, lot=0.01, horizons=(1, 2, 5, 12)):
    ordered = sorted(origin_results, key=lambda r: int(r['origin_idx']))
    n_orig = len(ordered)
    if n_orig == 0:
        return {'timeframe': tf_label, 'origins': 0, 'error': 'no_origins'}
    split_pos = n_orig // 2
    half_sp = spread_px * 0.5
    st_half_sp = spread_px * 0.75
    st_comm = 5.0 * lot
    h_reps = {}
    for h in horizons:
        k_abs, nc_abs = [], []
        dh_a = dn_a = dh_f = dn_f = 0
        ta, ta1, ta2, tas = [], [], [], []
        tf_, tf1, tf2, tfs = [], [], [], []
        for rank, item in enumerate(ordered):
            idx = int(item['origin_idx'])
            preds = item['pred_closes']
            if idx < 1 or idx + h - 1 >= len(bars) or len(preds) < h:
                continue
            last_c = float(bars[idx - 1][4])
            entry_o = float(bars[idx][1])
            act_c = float(bars[idx + h - 1][4])
            pred_c = float(preds[h - 1])
            if not all(math.isfinite(x) for x in (last_c, entry_o, act_c, pred_c)):
                continue
            k_abs.append(abs(pred_c - act_c))
            nc_abs.append(abs(last_c - act_c))
            delta = pred_c - last_c
            if delta == 0.0:
                continue
            d = 1 if delta > 0 else -1
            gross = (act_c - entry_o) * d
            dn_a += 1
            if gross > 0:
                dh_a += 1
            pnl = (gross - 2.0 * half_sp) * mult * lot
            pnl_st = (gross - 2.0 * st_half_sp) * mult * lot - st_comm
            rec = {'pnl': round(pnl, 4)}
            rec_st = {'pnl': round(pnl_st, 4)}
            ta.append(rec); tas.append(rec_st)
            (ta1 if rank < split_pos else ta2).append(rec)
            if abs(delta) >= 2.0 * spread_px:
                dn_f += 1
                if gross > 0:
                    dh_f += 1
                tf_.append(rec); tfs.append(rec_st)
                (tf1 if rank < split_pos else tf2).append(rec)
        k_mae = (sum(k_abs) / len(k_abs)) if k_abs else None
        nc_mae = (sum(nc_abs) / len(nc_abs)) if nc_abs else None
        ratio = (k_mae / nc_mae) if (k_mae is not None and nc_mae and nc_mae > 0) else None
        h_reps[f'h{h}'] = {
            'horizon_bars': h, 'n': len(k_abs),
            'kronos_mae': round(k_mae, 4) if k_mae is not None else None,
            'no_change_mae': round(nc_mae, 4) if nc_mae is not None else None,
            'mae_ratio_vs_no_change': round(ratio, 3) if ratio is not None else None,
            'beats_no_change': bool(ratio is not None and ratio < 1.0),
            'all_signals': {
                'dir_hit_pct': round(100.0 * dh_a / dn_a, 2) if dn_a else None,
                'trades': _summarize(ta), 'h1_net': _summarize(ta1)['net'],
                'h2_net': _summarize(ta2)['net'], 'stress_net': _summarize(tas)['net'],
            },
            'conviction_2x_spread': {
                'dir_hit_pct': round(100.0 * dh_f / dn_f, 2) if dn_f else None,
                'trades': _summarize(tf_), 'h1_net': _summarize(tf1)['net'],
                'h2_net': _summarize(tf2)['net'], 'stress_net': _summarize(tfs)['net'],
            },
        }
    return {
        'timeframe': tf_label, 'bars': len(bars), 'origins': n_orig,
        'spread_px': round(spread_px, 5), 'mult': round(mult, 4),
        'ohlc_invalid_candles': sum(int(r.get('ohlc_invalid_count', 0)) for r in ordered),
        'vol_neg_candles': sum(int(r.get('vol_neg_count', 0)) for r in ordered),
        'horizons': h_reps,
    }


t_all = time.monotonic()
tf_reports = []
for tf_label in ('M1', 'M5', 'M15', 'H1', 'H4'):
    bars = tf_map.get(tf_label) or []
    origins = _select_origins(bars, context_len=CONTEXT_LEN, pred_len=PRED_LEN, n_origins=N_ORIGINS)
    if not origins:
        tf_reports.append({'timeframe': tf_label, 'origins': 0, 'error': f'insufficient_bars({len(bars)})'})
        continue
    t_tf = time.monotonic()
    origin_results = []
    for k_i, idx in enumerate(origins):
        ctx = bars[idx - CONTEXT_LEN : idx]
        df = pd.DataFrame(ctx, columns=['time', 'open', 'high', 'low', 'close', 'volume'])
        x_ts = pd.Series(pd.to_datetime(df.pop('time'), unit='s', utc=True).dt.tz_localize(None))
        y_ts = pd.Series(pd.date_range(x_ts.iloc[-1] + pd.Timedelta(seconds=TF_SEC[tf_label]), periods=PRED_LEN, freq=TF_FREQ[tf_label]))
        res = predictor.predict(df=df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=PRED_LEN, sample_count=SAMPLE_COUNT, verbose=False)
        pred_closes = []
        ohlc_bad = 0
        vol_bad = 0
        for _, row in res.iterrows():
            o, h, l, c, v = [float(row[col]) for col in ('open', 'high', 'low', 'close', 'volume')]
            if h < max(o, c, l) or l > min(o, c, h):
                ohlc_bad += 1
            if v < 0:
                vol_bad += 1
            pred_closes.append(c)
        origin_results.append({
            'origin_idx': idx,
            'pred_closes': pred_closes,
            'ohlc_invalid_count': ohlc_bad,
            'vol_neg_count': vol_bad,
        })
    rep_tf = _eval_tf(tf_label, bars, origin_results, spread_px=spread_px, mult=mult, lot=LOT)
    tf_reports.append(rep_tf)
    print(f'Finished {tf_label} ({len(origins)} origins) in {time.monotonic() - t_tf:.1f}s')

report = {
    'schema': 'kronos-multitf-benchmark/v1',
    'source': 'official_KronosPredictor.predict',
    'kronos_commit': KRONOS_COMMIT,
    'model_id': MODEL_ID,
    'tokenizer_id': TOKENIZER_ID,
    'sample_count': SAMPLE_COUNT,
    'context_len': CONTEXT_LEN,
    'pred_len': PRED_LEN,
    'symbol': symbol,
    'spread_px': spread_px,
    'mult': mult,
    'research_only': True,
    'execution_authorized': False,
    'validated_for_trading': False,
    'elapsed_sec': round(time.monotonic() - t_all, 1),
    'timeframes': tf_reports,
}

print('\n=== KRONOS MULTI-TIMEFRAME BENCHMARK (M1 -> H4 | ZERO LEAKAGE | REAL ALPARI SPREAD) ===')
print(f"SYMBOL={symbol} MODEL={MODEL_ID} SAMPLES={SAMPLE_COUNT} SPREAD={spread_px} ELAPSED={report['elapsed_sec']}s")
for tf_rep in tf_reports:
    tf = tf_rep['timeframe']
    if 'error' in tf_rep:
        print(f"\n[{tf}] ERROR: {tf_rep['error']}")
        continue
    print(f"\n[{tf}] bars={tf_rep['bars']} origins={tf_rep['origins']} ohlc_bad={tf_rep['ohlc_invalid_candles']} vol_bad={tf_rep['vol_neg_candles']}")
    for hkey in ('h1', 'h2', 'h5', 'h12'):
        hr = tf_rep['horizons'].get(hkey)
        if not hr:
            continue
        al, fc = hr['all_signals'], hr['conviction_2x_spread']
        at, ft = al['trades'], fc['trades']
        rs = '  None' if hr['mae_ratio_vs_no_change'] is None else f"{hr['mae_ratio_vs_no_change']:6.3f}"
        pfa = ' None' if at['pf'] is None else f"{at['pf']:5.2f}"
        pff = ' None' if ft['pf'] is None else f"{ft['pf']:5.2f}"
        tag = 'BEAT' if hr['beats_no_change'] else 'FAIL'
        print(
            f"  +{hr['horizon_bars']:2d}B [{tag}] MAE={hr['kronos_mae']} vs NC={hr['no_change_mae']} "
            f"(ratio={rs}) | ALL: dir={al['dir_hit_pct']}% net=${at['net']:+7.2f} PF={pfa} "
            f"H1/H2=({al['h1_net']:+.1f}/{al['h2_net']:+.1f}) st=${al['stress_net']:+.1f} | "
            f"CONV(>=2sp,n={ft['trades']:2d}): dir={fc['dir_hit_pct']}% net=${ft['net']:+7.2f} PF={pff}"
        )

out_report = f'/content/kronos_multitf_report_{symbol}.json'
with open(out_report, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print('\nSaved report:', out_report)
files.download(out_report)
